# Stage 3.2 — the regression run

The suite executed a second time **with no configuration change**, and every
execution compared with its case's baseline reply (B11). Per D23:

- a structural `detected` fails the execution, whatever the judge says;
- otherwise the judge compares the two first replies in **both orderings**
  (the position-bias control), and the execution passes only when both
  orderings say the candidate is at least as good;
- a **seed** case passes only when every repetition passes; an **expanded**
  case passes at ≥ 75 %.

A comparison the judge could not make is a missing measurement. The case is
then `incomplete`, never passed.

Like `03_baseline`, this notebook is driven by its parameters. Pointed at a
test baseline, it says so.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))

import pandas as pd

from src import survey, comparison as cmp, db
from src.cases import load_cases
from src.llm import load_prompt
from src.scoring import scores_for

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 90)

# --- parameters -------------------------------------------------------------
# The candidate run. The value below is a pipeline rehearsal: a full n = 1 run
# of all 31 cases, compared against the TEST baseline 03_baseline wrote.
CANDIDATE_RUN_ID = "5f4f6841-88e6-4a65-8faa-a2b5a680c221"
BASELINE_FILE = cmp.RUNS_DIR / "baseline_test.json"
# ---------------------------------------------------------------------------

TEST = cmp.is_test_baseline(BASELINE_FILE)
baseline = cmp.load_baseline(BASELINE_FILE)
print("TEST BASELINE - a rehearsal of the comparison, not stage 3.2" if TEST else "ADOPTED BASELINE (cases/baseline.json)")
print(f"baseline: {len(baseline['cases'])} cases from run(s) {', '.join(r[:8] for r in baseline['runs'])}, "
      f"adopted by {baseline.get('adopted_by')} on {baseline.get('on')}")
print()
print(survey.headline(CANDIDATE_RUN_ID))
cases = {c.case_id: c for c in load_cases()}

TEST BASELINE - a rehearsal of the comparison, not stage 3.2
baseline: 31 cases from run(s) 287e7780, 42cab17a, adopted by agent (pipeline rehearsal) on 2026-09-24



run 5f4f6841  started 2026-09-24T18:12:32
  notes         FULL SUITE n=1 (31 cases, incl. D81 seeds) - pre-baseline verification, NOT a baseline
  agent         gemini-3.5-flash (released 2026-05-19)
  api           0.1.3 @ 5af96abf452b
  suite         a1d6a7f96d8d
  executions    31  (+0 superseded)
  failed        0
  scored        31, mean 9.35
  cost          $0.7492


## 1. Nothing changed but the run

B11 requires the second run with no configuration change. Every version field
of the candidate is compared with each run the baseline was drawn from. Any
difference is named: a changed model or prompt would make the comparison
measure the change, not the agent's consistency. `baseline_ref` is how the
record ties the two runs together (B11: the two run rows are distinguishable
by `baseline_ref` alone).

In [2]:
VERSION_FIELDS = ["model_version", "model_released_on", "prompt_version", "tool_version",
                  "suite_version", "api_version", "api_commit", "app_version",
                  "simulator_model", "simulator_prompt_version", "judge_model", "judge_prompt_version"]
row_of = lambda r: db.admin().table("study_runs").select("*").eq("id", r).execute().data[0]
cand = row_of(CANDIDATE_RUN_ID)
parity = pd.DataFrame({
    f"baseline {r[:8]}": {f: row_of(r)[f] == cand[f] for f in VERSION_FIELDS} for r in baseline["runs"]
})
parity.insert(0, "candidate", [cand[f] for f in VERSION_FIELDS])
changed = {f for f in VERSION_FIELDS if not parity.loc[f].iloc[1:].all()}
print(f"fields that differ from a baseline run: {sorted(changed) or 'none'}")
expected_ref = baseline["runs"][0] if len(baseline["runs"]) == 1 else None
print(f"candidate baseline_ref: {cand.get('baseline_ref') or 'unset'}; "
      f"expected: {expected_ref or 'n/a - the baseline spans runs (rehearsal only)'}")
print(f"pair-wise judge prompt: {load_prompt('judge_pairwise_v1').version}")
parity

fields that differ from a baseline run: ['suite_version']
candidate baseline_ref: unset; expected: n/a - the baseline spans runs (rehearsal only)
pair-wise judge prompt: judge-pairwise/1


,candidate,baseline 287e7780,baseline 42cab17a
model_version,gemini-3.5-flash,True,True
model_released_on,2026-05-19,True,True
prompt_version,3ca333b6763e,True,True
tool_version,2e6f598f22a1,True,True
suite_version,a1d6a7f96d8db36665b1d14088c4c5937bc2c77f5aabc6935c6df3785bb3ab9f,False,False
api_version,0.1.3,True,True
api_commit,5af96abf452b,True,True
app_version,0.1.1,True,True
simulator_model,gemini-3.5-flash,True,True
simulator_prompt_version,simulator/1,True,True


## 2. Coverage

Every case in the suite should appear in the candidate run and in the
baseline. A case missing from either drops out of the pass rates silently, so
it is named here instead.

In [3]:
cand_frame = survey.run_frame(CANDIDATE_RUN_ID)
in_run = set(cand_frame["case"])
gaps = {
    "suite cases not executed in the candidate": sorted(set(cases) - in_run),
    "suite cases without a baseline entry": cmp.baseline_gaps(baseline),
    "executed cases the suite no longer has": sorted(in_run - set(cases)),
}
for k, v in gaps.items():
    print(f"{k}: {v or 'none'}")

suite cases not executed in the candidate: none
suite cases without a baseline entry: none
executed cases the suite no longer has: none


## 3. The comparison

Each execution's first reply against its case's baseline reply, in both
orderings. Verdicts are cached under `runs/comparisons/`, so re-executing this
cell resumes rather than re-judges. `orderings` lists the judge's two answers:
the first with the candidate shown as B, the second with it shown as A.

In [4]:
comparisons = cmp.compare_run(CANDIDATE_RUN_ID, baseline, log=lambda m: None)
frame = cmp.comparison_frame(comparisons)
def agreement(o):
    if len(o) != 2:
        return "-"
    held = [o[0] in ("B", "tie"), o[1] in ("A", "tie")]
    return "both hold" if all(held) else ("both fail" if not any(held) else "orderings disagree")
frame["orderings agree"] = frame["orderings"].map(agreement)
print(f"{len(frame)} executions compared; judge errors: {int(frame['error'].notna().sum())}")
print(frame["orderings agree"].value_counts().to_string())
frame[["case_id", "repetition", "structural", "orderings", "orderings agree", "passed"]]

31 executions compared; judge errors: 0
orderings agree
both hold             24
both fail              4
orderings disagree     3


,case_id,repetition,structural,orderings,orderings agree,passed
0,advisory_packing_list,1,none,"[A, B]",both fail,False
1,advisory_packing_list__inv1_paraphrase,1,none,"[A, B]",both fail,False
2,advisory_packing_list__inv3_irrelevant_detail,1,none,"[B, A]",both hold,True
3,advisory_sintra_full_or_half_day,1,none,"[tie, tie]",both hold,True
4,advisory_sintra_full_or_half_day__inv1_paraphrase,1,none,"[tie, tie]",both hold,True
5,advisory_sintra_full_or_half_day__inv4_german,1,none,"[tie, tie]",both hold,True
6,factual_qa_alfama_history,1,none,"[tie, tie]",both hold,True
7,factual_qa_alfama_history__dir1_the_earthquake,1,none,"[tie, tie]",both hold,True
8,factual_qa_alfama_history__inv1_paraphrase,1,none,"[B, tie]",both hold,True
9,factual_qa_lisbon_forecast_next_days,1,none,"[B, A]",both hold,True


**Orderings that disagree** mean the judge preferred whichever reply was
shown in one position. D23 counts only both-hold as a pass, so position bias
can fail an execution but not pass one. Their share is the judge's own
variance on this run.

## 4. Pass rates and verdicts (D23)

In [5]:
rates = cmp.pass_rates(comparisons)
print(rates["verdict"].value_counts().to_string())
print()
rates.assign(**{"pass rate": rates["pass rate"].map("{:.0%}".format)})

verdict
pass    23
fail     8



,provenance,executions,passed,structural fails,judge says worse,not compared,pass rate,threshold,verdict
case,,,,,,,,,
advisory_packing_list,seeded,1,0,0,1,0,0%,1.00,fail
advisory_packing_list__inv1_paraphrase,metamorphic,1,0,0,1,0,0%,0.75,fail
advisory_packing_list__inv3_irrelevant_detail,metamorphic,1,1,0,0,0,100%,0.75,pass
advisory_sintra_full_or_half_day,seeded,1,1,0,0,0,100%,1.00,pass
advisory_sintra_full_or_half_day__inv1_paraphrase,metamorphic,1,1,0,0,0,100%,0.75,pass
advisory_sintra_full_or_half_day__inv4_german,metamorphic,1,1,0,0,0,100%,0.75,pass
factual_qa_alfama_history,seeded,1,1,0,0,0,100%,1.00,pass
factual_qa_alfama_history__dir1_the_earthquake,metamorphic,1,1,0,0,0,100%,0.75,pass
factual_qa_alfama_history__inv1_paraphrase,metamorphic,1,1,0,0,0,100%,0.75,pass


## 5. Why each failure failed

A structural fail cites the checker's findings; a judge fail cites both
orderings' rationale. The two are kept apart because they are different
evidence: the checker saw what the agent did, the judge only what it said.

In [6]:
execs = {e["id"]: e for e in cmp._executions([CANDIDATE_RUN_ID])}
failed = [c for c in comparisons if c.passed is False]
if not failed:
    print("no execution failed")
for c in failed:
    print("=" * 100)
    print(f"{c.case_id} rep {c.repetition}")
    if c.structural == "detected":
        for f in execs[c.execution_id].get("structural_findings") or []:
            print("  structural:", f)
    else:
        print(f"  judge {c.orderings}: {c.rationale[:500]}")

advisory_packing_list rep 1
  judge ['A', 'B']: Reply A is slightly better because it explicitly writes for the group, addressing the user's prompt about what 'the three of us' should pack. Additionally, its advice on rain protection is more tailored to the specific destination, correctly noting that Sintra's high winds make a hooded jacket a much better choice than an umbrella. | Reply B is slightly better because it explicitly addresses the group of three ("Here is what the three of us should pack") in response to the user's prompt. Additio
advisory_packing_list__inv1_paraphrase rep 1
  judge ['A', 'B']: Both replies are excellent and meet all the criteria, including referencing the specific weather, cobblestones, and Sintra hills. Reply A is slightly better because it explicitly writes for the group ("Here is what you three should pack") in response to the member's question about "the three of us." | Both replies do an excellent job of tailoring the packing list to the specific dest

## 6. Variability

§3.1: where variability is high, measures to reduce it are explored. Three
signals are available on every run. The point-wise score of each candidate
execution against its baseline's score (a large drop the pair-wise verdict
tolerated, or a pair-wise fail on equal scores, is worth reading). The spread
of scores across a case's repetitions (zero at `n = 1`, and meaningful only
at full `n`). And the orderings disagreement above.

In [7]:
base_scores = {}
for r in baseline["runs"]:
    for s in scores_for(r):
        base_scores.setdefault(s["execution_id"], s["score"])
cand_scores = {}
for s in scores_for(CANDIDATE_RUN_ID):
    cand_scores.setdefault(s["execution_id"], s["score"])
frame["candidate score"] = frame["execution_id"].map(cand_scores)
frame["baseline score"] = frame["baseline_execution_id"].map(base_scores)
frame["delta"] = frame["candidate score"] - frame["baseline score"]
spread = frame.groupby("case_id")["candidate score"].agg(["count", "std", "min", "max"])
print(f"candidate executions without a point-wise score: {int(frame['candidate score'].isna().sum())}")
print(f"largest score drops against the baseline:")
frame.sort_values("delta")[["case_id", "repetition", "baseline score", "candidate score", "delta", "passed"]].head(8)

candidate executions without a point-wise score: 0
largest score drops against the baseline:


,case_id,repetition,baseline score,candidate score,delta,passed
9,factual_qa_lisbon_forecast_next_days,1,10,7,-3,True
26,venue_lookup_seafood_lisbon__dir4_not_a_meal,1,10,8,-2,False
13,factual_qa_lisbon_surf_conditions,1,7,6,-1,True
0,advisory_packing_list,1,10,9,-1,False
17,scheduling_belem_morning__dir2_on_the_9th,1,10,9,-1,True
18,scheduling_belem_morning__dir3_a_full_day,1,10,9,-1,False
6,factual_qa_alfama_history,1,10,10,0,True
1,advisory_packing_list__inv1_paraphrase,1,10,10,0,False


## 7. Cost and latency, run against run

In [8]:
def totals(run_ids):
    f = pd.concat([survey.run_frame(r) for r in run_ids])
    return {"executions": len(f), "cost_usd": round(f["cost_usd"].sum(), 4),
            "cost per execution": round(f["cost_usd"].mean(), 4),
            "median latency s": round(f["latency_ms"].median() / 1000, 1)}
pd.DataFrame({"baseline": totals(baseline["runs"]), "candidate": totals([CANDIDATE_RUN_ID])})

,baseline,candidate
executions,35.0000,31.0000
cost_usd,0.9965,0.7492
cost per execution,0.0285,0.0242
median latency s,35.9000,23.1000


## 8. Done when (B11)

Computed, so a gap cannot hide in prose. A `False` is a gap to close.

In [9]:
done = {
    "every suite case was executed": not gaps["suite cases not executed in the candidate"],
    "every case has a baseline": not gaps["suite cases without a baseline entry"],
    "every execution was compared": frame["error"].isna().all(),
    "every execution was structurally checked": frame["structural"].isin(["none", "detected"]).all(),
    "every case has a verdict": (rates["verdict"] != "incomplete").all(),
    "no configuration change": not (changed - {"suite_version"} if TEST else changed),
    "baseline_ref ties the runs": bool(expected_ref) and cand.get("baseline_ref") == expected_ref,
    "the baseline is the adopted file": not TEST,
}
pd.Series(done, name="holds").to_frame()

,holds
every suite case was executed,True
every case has a baseline,True
every execution was compared,True
every execution was structurally checked,True
every case has a verdict,True
no configuration change,True
baseline_ref ties the runs,False
the baseline is the adopted file,False


## 9. Reading this run

*Rehearsal: the full n = 1 run `5f4f6841` (31 cases, including the two
multi-step seeds) against a test baseline drawn from `42cab17a` and the D82
probe. Not stage 3.2: every case ran once, and the baseline is not adopted.*

**The flow works end to end.** All 31 executions were compared, with no judge
errors. Both kinds of evidence reach the verdict: one structural fail and
seven judge fails. The configuration check caught the one field that differs
(`suite_version`, because the baseline runs predate the new seeds), and the
run survey finds no unexplained empty column. A self-comparison beforehand
(42cab17a against itself) returned `tie`/`tie` for all 29 identical pairs, so
swapping and caching do not invent differences.

**Eight of 31 cases fail, and seven of them on the judge alone.** This is the
number to understand before a full run:

- `scheduling_afternoon_then_tram` fails **structurally**. It is the
  duplication defect again (the lunch re-drafted at 12:30, 5 slots). A real
  finding about the agent.
- **Four fail with both orderings preferring the baseline**, on narrow
  stylistic grounds the judge itself calls "slightly better": addressing
  "the three of us" (the packing seed and its paraphrase), reciting the
  stops in the opening summary (Belém), confirming a change twice (the
  monastery). All of these candidates score 9–10 point-wise.
- **Three fail because the orderings disagree**. The judge preferred
  whichever reply sat in one position. D23 counts only both-hold as a pass,
  so position bias fails these executions: 3 of 31 here.

**What this predicts for the real runs.** D86 adopts each case's best
execution as its baseline. D23 requires a seed to hold in all eight
repetitions and an expanded case in three of four. At n = 1, three of 13
seeds already fail on the judge alone. At full n most seeds will meet at
least one narrowly weaker or position-biased repetition, and fail.
That is the design working as specified, not a harness defect. But it means a
seed verdict of `fail` will mostly say "not reliably as good as its best
reply", not "regressed". Whether that is the intended reading is a decision
(Q34).

**Point-wise and pair-wise can disagree.** `factual_qa_lisbon_forecast_next_days`
scored 7 point-wise against the baseline's 10, yet held pair-wise in both
orderings. Its two replies describe forecasts fetched on different days, so
the pair compares different weather. The scores are two separate instruments
and are reported side by side, never merged.

**Gaps this rehearsal leaves, by design or by decision:** `baseline_ref` is
unset (a test baseline spans two runs; the runner sets it for a real
regression run). The baseline file is a test file. Pair-wise verdicts live in
a local cache, not the database (Q33). Variability across repetitions cannot
be read at n = 1.